## Merge Notebook

This project will use 3 merged datasets:
1. ACS_CPS_Merged.csv: this contains the ACS and CPS and will be merged under [CBSA + Year + Age Group]
2. ACS_FMR_HPI_Merged.csv: this contains the ACS, FMR, and HPI and will be merged under [County + Year]
3. ACS_CPS_FMR_HPI_Merged.csv: this contains the ACS, CPS, FMR, and HPI will be merged under [CBSA + Year + Age Group]

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


### Step 1: Merge ACS with CPS

In [2]:
clean_path = Path("../clean_data")

acs_path = clean_path / "ACS_generation.csv"
cps_path = clean_path / "CLEANED_CPS_2005_2023.csv"

acs = pd.read_csv(acs_path, dtype={"cbsa_code": str})
cps = pd.read_csv(cps_path, dtype=str)

print("ACS shape:", acs.shape)
print("CPS shape:", cps.shape)

ACS shape: (1197, 10)
CPS shape: (52042, 14)


In [3]:
cps.head()

,State,County,Sub/Urb,HH_Size,Tenure,HH_Income,Supp_Weight,Relationship,Weight,Age,Education,Marital,Person_Income,Year
0,34,10900,2,2,1,149300,329433,1,3193.36,60,43,1,63300,2005
1,34,10900,2,2,1,56890,321250,1,2940.15,76,43,1,43847,2005
2,34,10900,2,2,1,209062,321250,1,3223.87,70,40,1,127959,2005
3,34,10900,2,4,1,131846,141104,1,3515.44,38,41,1,51346,2005
4,34,10900,2,2,1,57505,276382,1,3175.67,65,34,1,48649,2005


In [4]:
acs.head()

,cbsa_code,cbsa_name,year,age_group,owner_units_estimated,renter_units_estimated,total_age_group_units,age_group_share_of_total_units,owner_rate_within_age_group,renter_rate_within_age_group
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,15-24,153.0,181.0,334.0,0.008267,0.458084,0.541916
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,25-34,3577.0,2418.0,5995.0,0.148391,0.596664,0.403336
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,35-44,7083.0,2849.0,9932.0,0.245842,0.713149,0.286851
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,45-54,8140.0,2477.0,10617.0,0.262797,0.766695,0.233305
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,55-59,2918.0,1048.0,3966.0,0.098168,0.735754,0.264246


Since CPS has individual household/person rows, I need to group it so there is one row for each year, CBSA, and age group.


In [5]:
print("CPS years:")
print(sorted(cps["Year"].dropna().unique()))

print("\nCPS County/CBSA values:")
print(cps["County"].value_counts(dropna=False).head(20))

print("\nCPS Relationship values:")
print(cps["Relationship"].value_counts(dropna=False))

print("\nCPS Tenure values:")
print(cps["Tenure"].value_counts(dropna=False))

CPS years:
['2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']

CPS County/CBSA values:
County
35620    39176
37980     7256
12100     1820
45940     1814
47220      950
0          422
10900      360
36140      244
Name: count, dtype: int64

CPS Relationship values:
Relationship
1    37942
2    14100
Name: count, dtype: int64

CPS Tenure values:
Tenure
1    33580
2    18088
3      374
Name: count, dtype: int64


CPS needs to match the ACS row level before merging. I will rename the CPS location column to `cbsa_code`, keep the householder rows, and create broader age groups. The broader age groups reduce missing CPS matches while still keeping age detail.


In [6]:
cps_clean = cps.copy()

cps_clean = cps_clean.rename(columns={
    "County": "cbsa_code",
    "Year": "year"
})

cps_clean["cbsa_code"] = (
    pd.to_numeric(cps_clean["cbsa_code"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .replace("<NA>", np.nan)
    .str.zfill(5)
)

cps_clean["year"] = pd.to_numeric(cps_clean["year"], errors="coerce")
cps_clean["Age"] = pd.to_numeric(cps_clean["Age"], errors="coerce")
cps_clean["Tenure"] = pd.to_numeric(cps_clean["Tenure"], errors="coerce")
cps_clean["Relationship"] = pd.to_numeric(cps_clean["Relationship"], errors="coerce")
cps_clean["Supp_Weight"] = pd.to_numeric(cps_clean["Supp_Weight"], errors="coerce")

cps_clean = cps_clean[
    (cps_clean["cbsa_code"].notna()) &
    (cps_clean["cbsa_code"] != "00000") &
    (cps_clean["year"].notna()) &
    (cps_clean["Age"].notna()) &
    (cps_clean["Supp_Weight"].notna())
].copy()

cps_clean = cps_clean[cps_clean["Relationship"] == 1].copy()

cps_clean.head()


,State,cbsa_code,Sub/Urb,HH_Size,Tenure,HH_Income,Supp_Weight,Relationship,Weight,Age,Education,Marital,Person_Income,year
0,34,10900,2,2,1,149300,329433,1,3193.36,60,43,1,63300,2005
1,34,10900,2,2,1,56890,321250,1,2940.15,76,43,1,43847,2005
2,34,10900,2,2,1,209062,321250,1,3223.87,70,40,1,127959,2005
3,34,10900,2,4,1,131846,141104,1,3515.44,38,41,1,51346,2005
4,34,10900,2,2,1,57505,276382,1,3175.67,65,34,1,48649,2005


In [7]:
def assign_age_group(age):
    if 15 <= age <= 34:
        return "15-34"
    elif 35 <= age <= 44:
        return "35-44"
    elif 45 <= age <= 54:
        return "45-54"
    elif 55 <= age <= 64:
        return "55-64"
    elif 65 <= age <= 74:
        return "65-74"
    elif age >= 75:
        return "75+"
    else:
        return np.nan

cps_clean["age_group"] = cps_clean["Age"].apply(assign_age_group)
cps_clean = cps_clean[cps_clean["age_group"].notna()].copy()

cps_clean[["cbsa_code", "year", "Age", "age_group", "Tenure", "Supp_Weight"]].head()



,cbsa_code,year,Age,age_group,Tenure,Supp_Weight
0,10900,2005,60,55-64,1,329433
1,10900,2005,76,75+,1,321250
2,10900,2005,70,65-74,1,321250
3,10900,2005,38,35-44,1,141104
4,10900,2005,65,65-74,1,276382


Now I group CPS by CBSA, year, and age group. Owner/renter counts are summed using the CPS weight. Other CPS variables are kept as weighted averages or shares so they are not lost during the merge.


In [8]:
def weighted_average(values, weights):
    valid = values.notna() & weights.notna()
    if valid.sum() == 0 or weights[valid].sum() == 0:
        return np.nan
    return np.average(values[valid], weights=weights[valid])

cps_clean["owner_units_cps"] = np.where(
    cps_clean["Tenure"] == 1,
    cps_clean["Supp_Weight"],
    0
)

cps_clean["renter_units_cps"] = np.where(
    cps_clean["Tenure"].isin([2, 3]),
    cps_clean["Supp_Weight"],
    0
)

cps_clean["HH_Income"] = pd.to_numeric(cps_clean["HH_Income"], errors="coerce")
cps_clean["Person_Income"] = pd.to_numeric(cps_clean["Person_Income"], errors="coerce")
cps_clean["HH_Size"] = pd.to_numeric(cps_clean["HH_Size"], errors="coerce")
cps_clean["Education"] = pd.to_numeric(cps_clean["Education"], errors="coerce")
cps_clean["Marital"] = pd.to_numeric(cps_clean["Marital"], errors="coerce")

cps_clean["married"] = np.where(cps_clean["Marital"] == 1, 1, 0)

cps_grouped = (
    cps_clean
    .groupby(["cbsa_code", "year", "age_group"])
    .apply(lambda group: pd.Series({
        "owner_units_cps": group["owner_units_cps"].sum(),
        "renter_units_cps": group["renter_units_cps"].sum(),
        "total_units_cps": group["Supp_Weight"].sum(),
        "avg_hh_income_cps": weighted_average(group["HH_Income"], group["Supp_Weight"]),
        "avg_person_income_cps": weighted_average(group["Person_Income"], group["Supp_Weight"]),
        "avg_hh_size_cps": weighted_average(group["HH_Size"], group["Supp_Weight"]),
        "avg_age_cps": weighted_average(group["Age"], group["Supp_Weight"]),
        "avg_education_cps": weighted_average(group["Education"], group["Supp_Weight"]),
        "married_share_cps": weighted_average(group["married"], group["Supp_Weight"]),
        "cps_records": len(group)
    }))
    .reset_index()
)

cps_grouped["owner_rate_cps"] = cps_grouped["owner_units_cps"] / cps_grouped["total_units_cps"]
cps_grouped["renter_rate_cps"] = cps_grouped["renter_units_cps"] / cps_grouped["total_units_cps"]

cps_grouped.head()


/var/folders/_l/_85pbswx3gb9p0649jw7x4rm0000gn/T/ipykernel_55555/2138526620.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cps_clean


,cbsa_code,year,age_group,owner_units_cps,renter_units_cps,total_units_cps,avg_hh_income_cps,avg_person_income_cps,avg_hh_size_cps,avg_age_cps,avg_education_cps,married_share_cps,cps_records,owner_rate_cps,renter_rate_cps
0,10900,2005,35-44,912618.0,0.0,912618.0,122281.585469,53697.384303,2.618458,39.381542,43.072313,1.0,4.0,1.000000,0.000000
1,10900,2005,45-54,352858.0,308510.0,661368.0,57814.224734,52419.728420,3.533527,49.600582,40.600582,1.0,4.0,0.533527,0.466473
2,10900,2005,55-64,658866.0,0.0,658866.0,149300.000000,63300.000000,2.000000,60.000000,43.000000,1.0,2.0,1.000000,0.000000
3,10900,2005,65-74,1195264.0,0.0,1195264.0,138972.669486,91281.150722,2.000000,67.687691,37.225229,1.0,4.0,1.000000,0.000000
4,10900,2005,75+,642500.0,0.0,642500.0,56890.000000,43847.000000,2.000000,76.000000,43.000000,1.0,2.0,1.000000,0.000000


The merge uses `cbsa_code`, `year`, and `age_group`. Before merging, I group the ACS age groups into the same six broader age groups used for CPS: 15-34, 35-44, 45-54, 55-64, 65-74, and 75+.


In [9]:
acs_merge = acs.copy()
cps_merge = cps_grouped.copy()

def clean_cbsa_code(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .replace("<NA>", np.nan)
        .str.zfill(5)
    )

def clean_age_group(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace("–", "-", regex=False)
        .str.replace("—", "-", regex=False)
        .str.replace("_", "-", regex=False)
        .str.replace("85-plus", "85+", regex=False)
        .str.replace("85 plus", "85+", regex=False)
        .str.replace("85 and over", "85+", regex=False)
    )

age_group_map = {
    "15-24": "15-34",
    "25-34": "15-34",
    "35-44": "35-44",
    "45-54": "45-54",
    "55-59": "55-64",
    "60-64": "55-64",
    "65-74": "65-74",
    "75-84": "75+",
    "85+": "75+"
}

acs_merge["cbsa_code"] = clean_cbsa_code(acs_merge["cbsa_code"])
cps_merge["cbsa_code"] = clean_cbsa_code(cps_merge["cbsa_code"])

acs_merge["year"] = pd.to_numeric(acs_merge["year"], errors="coerce").astype("Int64")
cps_merge["year"] = pd.to_numeric(cps_merge["year"], errors="coerce").astype("Int64")

acs_merge["age_group"] = clean_age_group(acs_merge["age_group"]).map(age_group_map)
cps_merge["age_group"] = clean_age_group(cps_merge["age_group"])

acs_merge = (
    acs_merge
    .groupby(["cbsa_code", "cbsa_name", "year", "age_group"], as_index=False)
    .agg({
        "owner_units_estimated": "sum",
        "renter_units_estimated": "sum",
        "total_age_group_units": "sum"
    })
)

cbsa_year_total = acs_merge.groupby(["cbsa_code", "year"])["total_age_group_units"].transform("sum")

acs_merge["age_group_share_of_total_units"] = acs_merge["total_age_group_units"] / cbsa_year_total
acs_merge["owner_rate_within_age_group"] = acs_merge["owner_units_estimated"] / acs_merge["total_age_group_units"]
acs_merge["renter_rate_within_age_group"] = acs_merge["renter_units_estimated"] / acs_merge["total_age_group_units"]

acs_cps_merged = acs_merge.merge(
    cps_merge,
    on=["cbsa_code", "year", "age_group"],
    how="left"
)

print("ACS columns:", acs_merge.shape[1])
print("CPS columns:", cps_merge.shape[1])
print("Merged shape:", acs_cps_merged.shape)

acs_cps_merged.head()



ACS columns: 10
CPS columns: 15
Merged shape: (798, 22)


,cbsa_code,cbsa_name,year,age_group,owner_units_estimated,renter_units_estimated,total_age_group_units,age_group_share_of_total_units,owner_rate_within_age_group,renter_rate_within_age_group,...,total_units_cps,avg_hh_income_cps,avg_person_income_cps,avg_hh_size_cps,avg_age_cps,avg_education_cps,married_share_cps,cps_records,owner_rate_cps,renter_rate_cps
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,15-34,3730.0,2599.0,6329.0,0.156658,0.589351,0.410649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,35-44,7083.0,2849.0,9932.0,0.245842,0.713149,0.286851,...,912618.0,122281.585469,53697.384303,2.618458,39.381542,43.072313,1.0,4.0,1.000000,0.000000
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,45-54,8140.0,2477.0,10617.0,0.262797,0.766695,0.233305,...,661368.0,57814.224734,52419.728420,3.533527,49.600582,40.600582,1.0,4.0,0.533527,0.466473
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,55-64,6093.0,1048.0,7141.0,0.176757,0.853242,0.146758,...,658866.0,149300.000000,63300.000000,2.000000,60.000000,43.000000,1.0,2.0,1.000000,0.000000
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,65-74,2700.0,0.0,2700.0,0.066832,1.000000,0.000000,...,1195264.0,138972.669486,91281.150722,2.000000,67.687691,37.225229,1.0,4.0,1.000000,0.000000


In [10]:
missing_summary = (
    acs_cps_merged
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]


renter_units_cps         157
total_units_cps          157
owner_rate_cps           157
cps_records              157
married_share_cps        157
avg_education_cps        157
avg_age_cps              157
avg_hh_size_cps          157
avg_person_income_cps    157
avg_hh_income_cps        157
renter_rate_cps          157
owner_units_cps          157
dtype: int64

In [11]:
acs_cps_merged.to_csv(clean_path / "ACS_CPS_merged.csv", index=False)


### Step 2: Merge ACS, FMR, and HPI

In [12]:
acs = pd.read_csv("../clean_data/ACS_county.csv")
fmr = pd.read_csv("../clean_data/FMR_clean.csv")
hpi = pd.read_csv("../clean_data/cleaned_hpi_2005-2023.csv", header=None)

hpi.columns = [
    "state",
    "county",
    "county_fips",
    "year",
    "annual_change_pct",
    "hpi",
    "hpi_1990_base",
    "hpi_2000_base"
]

print("ACS shape:", acs.shape)
print("FMR shape:", fmr.shape)
print("HPI shape:", hpi.shape)

ACS shape: (399, 8)
FMR shape: (399, 9)
HPI shape: (399, 8)


In [13]:
acs_merge = acs[
    [
        "county_fips",
        "county",
        "year",
        "total_occupied_units",
        "owner_occupied_total",
        "renter_occupied_total",
        "owner_rate",
        "renter_rate"
    ]
].copy()

fmr_merge = fmr[
    [
        "county_fips",
        "year",
        "fmr_0br",
        "fmr_1br",
        "fmr_2br",
        "fmr_3br",
        "fmr_4br"
    ]
].copy()

hpi_merge = hpi[
    [
        "county_fips",
        "year",
        "annual_change_pct",
        "hpi"
    ]
].copy()

In [14]:
for df in [acs_merge, fmr_merge, hpi_merge]:
    df["county_fips"] = pd.to_numeric(df["county_fips"], errors="coerce").astype("Int64")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

In [15]:
acs_fmr_hpi = (
    acs_merge
    .merge(fmr_merge, on=["county_fips", "year"], how="left")
    .merge(hpi_merge, on=["county_fips", "year"], how="left")
)

print("Merged shape:", acs_fmr_hpi.shape)

acs_fmr_hpi.head()

Merged shape: (399, 15)


,county_fips,county,year,total_occupied_units,owner_occupied_total,renter_occupied_total,owner_rate,renter_rate,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,annual_change_pct,hpi
0,34001,Atlantic,2005,101584,68970,32614,0.678946,0.321054,669,702,845,1101,1173,20.81,771.58
1,34003,Bergen,2005,332223,225175,107048,0.677783,0.322217,906,990,1132,1428,1729,16.34,952.01
2,34005,Burlington,2005,164046,129201,34845,0.787590,0.212410,663,761,914,1095,1328,16.17,691.44
3,34007,Camden,2005,190659,130576,60083,0.684867,0.315133,663,761,914,1095,1328,16.04,545.32
4,34009,Cape May,2005,43616,34101,9515,0.781846,0.218154,669,702,845,1101,1173,21.99,951.10


In [16]:
acs_fmr_hpi.to_csv(clean_path / "ACS_FMR_HPI_merged.csv", index=False)

### Step 3: Merge ACS, CPS, FMR, and HPI

We will need to chnage the location vlaues of FMR and HPI from county to cbsa

In [17]:
fmr = pd.read_csv("../clean_data/FMR_clean.csv")
hpi = pd.read_csv("../clean_data/cleaned_hpi_2005-2023.csv", header=None)

hpi.columns = [
    "state",
    "county",
    "county_fips",
    "year",
    "annual_change_pct",
    "hpi",
    "hpi_1990_base",
    "hpi_2000_base"
]

print("FMR shape:", fmr.shape)
print("HPI shape:", hpi.shape)

display(fmr.head())
display(hpi.head())

FMR shape: (399, 9)
HPI shape: (399, 8)


,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,fmr_percentile
0,34001,Atlantic,2005,669,702,845,1101,1173,40
1,34003,Bergen,2005,906,990,1132,1428,1729,50
2,34005,Burlington,2005,663,761,914,1095,1328,50
3,34007,Camden,2005,663,761,914,1095,1328,50
4,34009,Cape May,2005,669,702,845,1101,1173,40


,state,county,county_fips,year,annual_change_pct,hpi,hpi_1990_base,hpi_2000_base
0,NJ,Atlantic,34001,2005,20.81,771.58,220.46,188.59
1,NJ,Atlantic,34001,2006,12.56,868.49,248.15,212.27
2,NJ,Atlantic,34001,2007,-0.67,862.63,246.48,210.84
3,NJ,Atlantic,34001,2008,-5.51,815.09,232.89,199.22
4,NJ,Atlantic,34001,2009,-10.26,731.47,209.00,178.79


In [18]:
fmr_merge = fmr[
    [
        "county_fips",
        "county",
        "year",
        "fmr_0br",
        "fmr_1br",
        "fmr_2br",
        "fmr_3br",
        "fmr_4br"
    ]
].copy()

hpi_merge = hpi[
    [
        "county_fips",
        "county",
        "year",
        "annual_change_pct",
        "hpi"
    ]
].copy()

fmr_merge["county_fips"] = pd.to_numeric(fmr_merge["county_fips"], errors="coerce").astype("Int64")
hpi_merge["county_fips"] = pd.to_numeric(hpi_merge["county_fips"], errors="coerce").astype("Int64")

fmr_merge["year"] = pd.to_numeric(fmr_merge["year"], errors="coerce").astype("Int64")
hpi_merge["year"] = pd.to_numeric(hpi_merge["year"], errors="coerce").astype("Int64")

print("FMR merge shape:", fmr_merge.shape)
print("HPI merge shape:", hpi_merge.shape)

display(fmr_merge.head())
display(hpi_merge.head())

FMR merge shape: (399, 8)
HPI merge shape: (399, 5)


,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br
0,34001,Atlantic,2005,669,702,845,1101,1173
1,34003,Bergen,2005,906,990,1132,1428,1729
2,34005,Burlington,2005,663,761,914,1095,1328
3,34007,Camden,2005,663,761,914,1095,1328
4,34009,Cape May,2005,669,702,845,1101,1173


,county_fips,county,year,annual_change_pct,hpi
0,34001,Atlantic,2005,20.81,771.58
1,34001,Atlantic,2006,12.56,868.49
2,34001,Atlantic,2007,-0.67,862.63
3,34001,Atlantic,2008,-5.51,815.09
4,34001,Atlantic,2009,-10.26,731.47


In [22]:
county_to_cbsa = pd.DataFrame({
    "county_fips": [
        34001, 34003, 34005, 34007, 34009, 34011, 34013,
        34015, 34017, 34019, 34021, 34023, 34025, 34027,
        34029, 34031, 34033, 34035, 34037, 34039, 34041
    ],
    "cbsa_code": [
        "12100", "35620", "37980", "37980", "36140", "47220", "35620",
        "37980", "35620", "35620", "45940", "35620", "35620", "35620",
        "35620", "35620", "37980", "35620", "35620", "35620", "10900"
    ],
    "cbsa_name": [
        "Atlantic City-Hammonton, NJ",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
        "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
        "Ocean City, NJ",
        "Vineland-Bridgeton, NJ",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "Trenton-Princeton, NJ",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "New York-Newark-Jersey City, NY-NJ-PA",
        "Allentown-Bethlehem-Easton, PA-NJ"
    ]
})

fmr_cbsa_ready = fmr_merge.merge(
    county_to_cbsa,
    on="county_fips",
    how="left"
)

hpi_cbsa_ready = hpi_merge.merge(
    county_to_cbsa,
    on="county_fips",
    how="left"
)



display(fmr_cbsa_ready.head())
display(hpi_cbsa_ready.head())

,county_fips,county,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,cbsa_code,cbsa_name
0,34001,Atlantic,2005,669,702,845,1101,1173,12100,"Atlantic City-Hammonton, NJ"
1,34003,Bergen,2005,906,990,1132,1428,1729,35620,"New York-Newark-Jersey City, NY-NJ-PA"
2,34005,Burlington,2005,663,761,914,1095,1328,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"
3,34007,Camden,2005,663,761,914,1095,1328,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"
4,34009,Cape May,2005,669,702,845,1101,1173,36140,"Ocean City, NJ"


,county_fips,county,year,annual_change_pct,hpi,cbsa_code,cbsa_name
0,34001,Atlantic,2005,20.81,771.58,12100,"Atlantic City-Hammonton, NJ"
1,34001,Atlantic,2006,12.56,868.49,12100,"Atlantic City-Hammonton, NJ"
2,34001,Atlantic,2007,-0.67,862.63,12100,"Atlantic City-Hammonton, NJ"
3,34001,Atlantic,2008,-5.51,815.09,12100,"Atlantic City-Hammonton, NJ"
4,34001,Atlantic,2009,-10.26,731.47,12100,"Atlantic City-Hammonton, NJ"


In [21]:
fmr_cbsa = (
    fmr_cbsa_ready
    .groupby(["cbsa_code", "cbsa_name", "year"], as_index=False)
    .agg({
        "fmr_0br": "mean",
        "fmr_1br": "mean",
        "fmr_2br": "mean",
        "fmr_3br": "mean",
        "fmr_4br": "mean"
    })
)

hpi_cbsa = (
    hpi_cbsa_ready
    .groupby(["cbsa_code", "cbsa_name", "year"], as_index=False)
    .agg({
        "annual_change_pct": "mean",
        "hpi": "mean"
    })
)

print("FMR CBSA shape:", fmr_cbsa.shape)
print("HPI CBSA shape:", hpi_cbsa.shape)

display(fmr_cbsa.head())
display(hpi_cbsa.head())

FMR CBSA shape: (133, 8)
HPI CBSA shape: (133, 5)


,cbsa_code,cbsa_name,year,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,735.0,891.0,1020.0,1242.0,1403.0
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2006,763.0,854.0,999.0,1196.0,1231.0
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2007,802.0,898.0,1050.0,1257.0,1294.0
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2008,769.0,861.0,1007.0,1205.0,1241.0
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2009,790.0,884.0,1034.0,1238.0,1274.0


,cbsa_code,cbsa_name,year,annual_change_pct,hpi
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,13.55,700.59
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2006,8.75,761.92
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2007,-2.13,745.66
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2008,-5.30,706.16
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2009,-9.31,640.40


In [24]:
acs_cps_fmr_hpi = (
    acs_cps_merged
    .merge(
        fmr_cbsa.drop(columns=["cbsa_name"]),
        on=["cbsa_code", "year"],
        how="left"
    )
    .merge(
        hpi_cbsa.drop(columns=["cbsa_name"]),
        on=["cbsa_code", "year"],
        how="left"
    )
)

print("ACS+CPS shape:", acs_cps_merged.shape)
print("Final merged shape:", acs_cps_fmr_hpi.shape)

acs_cps_fmr_hpi.head()

ACS+CPS shape: (798, 22)
Final merged shape: (798, 29)


,cbsa_code,cbsa_name,year,age_group,owner_units_estimated,renter_units_estimated,total_age_group_units,age_group_share_of_total_units,owner_rate_within_age_group,renter_rate_within_age_group,...,cps_records,owner_rate_cps,renter_rate_cps,fmr_0br,fmr_1br,fmr_2br,fmr_3br,fmr_4br,annual_change_pct,hpi
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,15-34,3730.0,2599.0,6329.0,0.156658,0.589351,0.410649,...,NaN,NaN,NaN,735.0,891.0,1020.0,1242.0,1403.0,13.55,700.59
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,35-44,7083.0,2849.0,9932.0,0.245842,0.713149,0.286851,...,4.0,1.000000,0.000000,735.0,891.0,1020.0,1242.0,1403.0,13.55,700.59
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,45-54,8140.0,2477.0,10617.0,0.262797,0.766695,0.233305,...,4.0,0.533527,0.466473,735.0,891.0,1020.0,1242.0,1403.0,13.55,700.59
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,55-64,6093.0,1048.0,7141.0,0.176757,0.853242,0.146758,...,2.0,1.000000,0.000000,735.0,891.0,1020.0,1242.0,1403.0,13.55,700.59
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,65-74,2700.0,0.0,2700.0,0.066832,1.000000,0.000000,...,4.0,1.000000,0.000000,735.0,891.0,1020.0,1242.0,1403.0,13.55,700.59


In [ ]:
acs_cps_fmr_hpi.to_csv(
    "../clean_data/ACS_CPS_FMR_HPI_merged.csv",
    index=False
)